In [43]:
import os
import getpass
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage,AIMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import START, MessageGraph,StateGraph
from langchain_groq import chat_models,ChatGroq
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate


In [30]:
load_dotenv()


True

In [31]:
os.environ["LANGSMITH_TRACING"] = "true"
if not os.environ.get("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_API_KEY"] = getpass.getpass()

In [32]:
print("GROQ_API_KEY:", os.getenv("GROQ_API_KEY"))
print("LANGSMITH_API_KEY:", os.getenv("LANGSMITH_API_KEY"))
print("LANGSMITH_PROJECT:", os.getenv("LANGSMITH_PROJECT"))
print("LANGSMITH_TRACING:", os.getenv("LANGSMITH_TRACING"))
print("LANGSMITH_ENDPOINT:", os.getenv("LANGSMITH_ENDPOINT"))

GROQ_API_KEY: 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
LANGSMITH_API_KEY: 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
LANGSMITH_PROJECT: RAG_chatbot
LANGSMITH_TRACING: true
LANGSMITH_ENDPOINT: https://api.smith.langchain.com


In [33]:
model = ChatGroq(groq_api_key=os.getenv("GROQ_API_KEY"),model_name="llama3-8b-8192")

In [34]:
response = model.invoke("How are you ?")

In [35]:
print(response)

content="I'm just a language model, I don't have emotions or feelings like humans do. However, I'm functioning properly and ready to help with any questions or tasks you may have. How can I assist you today?" additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 45, 'prompt_tokens': 14, 'total_tokens': 59, 'completion_time': 0.0375, 'prompt_time': 0.002968916, 'queue_time': 0.23818545800000002, 'total_time': 0.040468916}, 'model_name': 'llama3-8b-8192', 'system_fingerprint': 'fp_dadc9d6142', 'finish_reason': 'stop', 'logprobs': None} id='run-d02812d4-9a6c-4ff7-b5b1-e0560c451b8a-0' usage_metadata={'input_tokens': 14, 'output_tokens': 45, 'total_tokens': 59}


In [36]:
model.invoke([HumanMessage(content="Hi, I'm Vivek Soni,How about you ?")])

AIMessage(content="Nice to meet you, Vivek! I'm LLaMA, I'm a large language model trained by a team of researcher at Meta AI. I don't have a personal identity, but I'm here to assist you with any questions or topics you'd like to discuss. How can I help you today?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 64, 'prompt_tokens': 23, 'total_tokens': 87, 'completion_time': 0.053333333, 'prompt_time': 0.005618328, 'queue_time': 2.0109093799999997, 'total_time': 0.058951661}, 'model_name': 'llama3-8b-8192', 'system_fingerprint': 'fp_179b0f92c9', 'finish_reason': 'stop', 'logprobs': None}, id='run-d285323c-087c-47e3-8329-091cd6606c98-0', usage_metadata={'input_tokens': 23, 'output_tokens': 64, 'total_tokens': 87})

## keep conversation history along with the query to get proper output.

In [39]:
from langchain_core.messages import AIMessage

llm_response = model.invoke(
    [
        HumanMessage(content="Hi, I'm Vivek Soni,How about you ?"),
        AIMessage(content="Nice to meet you, Vivek Soni!"),
        HumanMessage(content="What's my name?")
    ]
)

In [40]:
output_parser = StrOutputParser()
output_parser.invoke(llm_response)

'Your name is Vivek Soni!'

## Chains 

In [42]:
chain = model | output_parser
chain.invoke("Hello, I'm Vivek Soni!, how are you ?")

"Nice to meet you, Vivek Soni! I'm doing well, thanks for asking. I'm an AI, so I don't have emotions like humans do, but I'm always happy to chat with someone new. How about you? What brings you here today?"

In [46]:
template = ChatPromptTemplate([
    'System','You are a doctor. You can provide a description of a medicine based on the symptoms a particular person has',
    'human','I\'n having {symptoms}'
])

In [47]:
template.invoke({"symptoms":["headache","feeling unwell","Feeling sleepy whole day"]})

ChatPromptValue(messages=[HumanMessage(content='System', additional_kwargs={}, response_metadata={}), HumanMessage(content='You are a doctor. You can provide a description of a medicine based on the symptoms a particular person has', additional_kwargs={}, response_metadata={}), HumanMessage(content='human', additional_kwargs={}, response_metadata={}), HumanMessage(content="I'n having ['headache', 'feeling unwell', 'Feeling sleepy whole day']", additional_kwargs={}, response_metadata={})])

In [48]:
chain = template | model | output_parser
chain.invoke({"symptoms":["headache","feeling unwell","Feeling sleepy whole day"]})

"Based on your symptoms, I would like to narrow down the possible causes and consider a few potential medications that might help alleviate your discomfort.\n\nYour symptoms of headache, feeling unwell, and excessive sleepiness suggest that you might be experiencing a viral infection such as a cold or flu, which can cause fatigue, headaches, and a general feeling of being unwell.\n\nConsidering these symptoms, I would like to recommend a medication that can help alleviate your symptoms and provide relief from your headache and fatigue.\n\nI would like to prescribe you a medication called acetaminophen (Tylenol) 325mg every 4-6 hours as needed for your headache and fever. Additionally, I would like to recommend a medication called diphenhydramine (Benadryl) 25mg every 6-8 hours as needed to help with your sleepiness and promote a good night's sleep.\n\nPlease note that these medications are only prescribed for a short-term relief and it is important to consult with your doctor if your s

In [49]:
from langchain.document_loaders import TextLoader,PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

loader = PyPDFLoader("../SourceFile/Stock-Investing-101-eBook.pdf")
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = text_splitter.split_documents(documents)

Ignoring wrong pointing object 18 0 (offset 0)
Ignoring wrong pointing object 149 0 (offset 0)
Ignoring wrong pointing object 299 0 (offset 0)
Ignoring wrong pointing object 489 0 (offset 0)
Ignoring wrong pointing object 491 0 (offset 0)


In [ ]:
from langchain_community.vectorstores import Chroma
from langchain.embeddings.openai import OpenAIEmbeddings

# Initialize embeddings model
embedding_model = OpenAIEmbeddings()

# Store embeddings in ChromaDB
vectorstore = Chroma.from_documents(docs, embedding_model)
retriever = vectorstore.as_retriever()

ImportError: Could not import chromadb python package. Please install it with `pip install chromadb`.

In [ ]:
rag_chain = ConversationalRetrievalChain.from_llm(llm, retriever=retriever)

# Chat history for maintaining conversation
chat_history = []

In [ ]:
while True:
    query = input("You: ")
    
    if query.lower() in ["exit", "quit"]:
        print("Chatbot: Goodbye! 👋")
        break
    
    response = rag_chain.invoke({"question": query, "chat_history": chat_history})
    
    print(f"Chatbot: {response['answer']}")
    
    chat_history.append((query, response['answer']))  # Maintain history
